# GPU加速计算

本notebook介绍如何使用GPU加速深度学习计算:
- GPU vs CPU的性能对比
- 查询和选择GPU设备
- 在GPU上创建和计算张量
- 在GPU上训练神经网络
- 多GPU并行训练基础

GPU是深度学习的核心加速器!

## 第一部分: GPU基础知识

### 1.1 为什么需要GPU?

**GPU vs CPU**:

| 特性 | CPU | GPU |
|------|-----|-----|
| 核心数量 | 少(4-64) | 多(数千) |
| 单核性能 | 强 | 弱 |
| 并行能力 | 弱 | 强 |
| 适用场景 | 通用计算 | 大规模并行计算 |
| 深度学习训练 | 慢 | 快(10-100倍) |

**深度学习为什么适合GPU?**
- 矩阵运算可以高度并行化
- 训练需要大量重复计算
- GPU的高带宽内存加速数据传输

### 1.2 查看GPU信息

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import time

In [ ]:
# 查看PyTorch是否支持CUDA
print(f'PyTorch版本: {torch.__version__}')
print(f'CUDA是否可用: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'CUDA版本: {torch.version.cuda}')
    print(f'GPU数量: {torch.cuda.device_count()}')
    print(f'当前GPU: {torch.cuda.current_device()}')
    print(f'GPU名称: {torch.cuda.get_device_name(0)}')
else:
    print('\n注意: 没有检测到GPU,将使用CPU进行演示')
    print('在Google Colab中,可以通过 运行时 → 更改运行时类型 → 硬件加速器 → GPU 来启用GPU')

**在Google Colab中查看GPU信息**

In [ ]:
# 使用nvidia-smi命令(仅在有GPU的系统上可用)
# 取消注释下面一行来运行
# !nvidia-smi

---

## 第二部分: 计算设备

### 2.1 指定设备

In [ ]:
# CPU设备
cpu = torch.device('cpu')
print(f'CPU设备: {cpu}')

# GPU设备
if torch.cuda.is_available():
    gpu = torch.device('cuda')  # 默认是cuda:0
    gpu0 = torch.device('cuda:0')  # 第一块GPU
    print(f'GPU设备: {gpu}')
    print(f'GPU 0: {gpu0}')
    
    # 如果有多块GPU
    if torch.cuda.device_count() > 1:
        gpu1 = torch.device('cuda:1')  # 第二块GPU
        print(f'GPU 1: {gpu1}')

### 2.2 辅助函数

In [ ]:
def try_gpu(i=0):
    """如果存在,则返回gpu(i),否则返回cpu()"""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

def try_all_gpus():
    """返回所有可用的GPU,如果没有GPU,则返回[cpu()]"""
    devices = [torch.device(f'cuda:{i}')
               for i in range(torch.cuda.device_count())]
    return devices if devices else [torch.device('cpu')]

print(f'第一个设备: {try_gpu()}')
print(f'第十个设备(可能不存在): {try_gpu(10)}')
print(f'所有设备: {try_all_gpus()}')

---

## 第三部分: 张量与GPU

### 3.1 查询张量所在设备

In [ ]:
# 默认在CPU上创建
x = torch.tensor([1, 2, 3])
print(f'张量x所在设备: {x.device}')

### 3.2 在GPU上创建张量

In [ ]:
# 方法1: 创建时指定device
if torch.cuda.is_available():
    x_gpu = torch.ones(3, 4, device=try_gpu())
    print(f'在GPU上创建的张量: {x_gpu.device}')
    print(x_gpu)

In [ ]:
# 方法2: 使用cuda()方法
x_cpu = torch.rand(3, 4)
print(f'CPU上的张量: {x_cpu.device}')

if torch.cuda.is_available():
    x_gpu = x_cpu.cuda()  # 移动到GPU
    print(f'移动到GPU后: {x_gpu.device}')

In [ ]:
# 方法3: 使用to()方法(推荐)
device = try_gpu()
x = torch.rand(3, 4)
x = x.to(device)
print(f'使用to()移动后: {x.device}')

### 3.3 在不同设备之间复制

In [ ]:
if torch.cuda.is_available():
    # GPU → CPU
    x_gpu = torch.ones(3, 4, device=try_gpu())
    x_cpu = x_gpu.cpu()
    print(f'从GPU复制到CPU: {x_cpu.device}')
    
    # CPU → GPU
    x_gpu2 = x_cpu.to(try_gpu())
    print(f'从CPU复制到GPU: {x_gpu2.device}')
    
    # 检查是否是同一个对象
    print(f'\nx_gpu和x_gpu2是同一个对象吗? {x_gpu is x_gpu2}')
    print(f'值相等吗? {(x_gpu == x_gpu2).all()}')

### 3.4 GPU上的运算

In [ ]:
if torch.cuda.is_available():
    device = try_gpu()
    
    # 在GPU上创建张量
    A = torch.randn(1000, 1000, device=device)
    B = torch.randn(1000, 1000, device=device)
    
    # 在GPU上进行矩阵乘法
    C = torch.matmul(A, B)
    
    print(f'A在: {A.device}')
    print(f'B在: {B.device}')
    print(f'C在: {C.device}')
    print(f'\nC的形状: {C.shape}')
else:
    print('没有GPU,跳过GPU运算演示')

**注意**: 不同设备的张量不能直接运算!

In [ ]:
if torch.cuda.is_available():
    x_cpu = torch.ones(3, 4)
    x_gpu = torch.ones(3, 4, device=try_gpu())
    
    try:
        # 这会报错!
        z = x_cpu + x_gpu
    except RuntimeError as e:
        print(f'错误: {e}')
    
    # 正确做法: 先移动到同一设备
    z = x_cpu.to(try_gpu()) + x_gpu
    print(f'\n正确运算后的结果在: {z.device}')

---

## 第四部分: GPU vs CPU性能对比

### 4.1 矩阵乘法性能

In [ ]:
def benchmark_matmul(size=2000, device='cpu', num_runs=5):
    """测试矩阵乘法性能"""
    device = torch.device(device)
    
    # 创建随机矩阵
    A = torch.randn(size, size, device=device)
    B = torch.randn(size, size, device=device)
    
    # GPU需要同步
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # 预热
    for _ in range(3):
        _ = torch.matmul(A, B)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # 计时
    start = time.time()
    for _ in range(num_runs):
        C = torch.matmul(A, B)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    elapsed = time.time() - start
    avg_time = elapsed / num_runs
    
    return avg_time

# CPU性能
print('测试矩阵乘法性能(2000×2000)...')
cpu_time = benchmark_matmul(size=2000, device='cpu', num_runs=3)
print(f'CPU时间: {cpu_time:.4f}秒')

# GPU性能
if torch.cuda.is_available():
    gpu_time = benchmark_matmul(size=2000, device='cuda', num_runs=3)
    print(f'GPU时间: {gpu_time:.4f}秒')
    print(f'\n加速比: {cpu_time / gpu_time:.2f}x')
else:
    print('\n没有GPU,无法进行对比')

### 4.2 不同规模的性能对比

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if torch.cuda.is_available():
    sizes = [100, 500, 1000, 2000, 4000]
    cpu_times = []
    gpu_times = []
    
    print('测试不同矩阵大小的性能...')
    for size in sizes:
        print(f'  大小: {size}×{size}', end='')
        cpu_t = benchmark_matmul(size, 'cpu', num_runs=1)
        gpu_t = benchmark_matmul(size, 'cuda', num_runs=1)
        cpu_times.append(cpu_t)
        gpu_times.append(gpu_t)
        print(f' → CPU: {cpu_t:.4f}s, GPU: {gpu_t:.4f}s, 加速: {cpu_t/gpu_t:.1f}x')
    
    # 可视化
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(sizes, cpu_times, marker='o', label='CPU', linewidth=2)
    plt.plot(sizes, gpu_times, marker='s', label='GPU', linewidth=2)
    plt.xlabel('矩阵大小', fontsize=12)
    plt.ylabel('时间(秒)', fontsize=12)
    plt.title('CPU vs GPU 性能对比', fontsize=14)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    speedup = [c/g for c, g in zip(cpu_times, gpu_times)]
    plt.plot(sizes, speedup, marker='o', color='green', linewidth=2)
    plt.xlabel('矩阵大小', fontsize=12)
    plt.ylabel('加速比', fontsize=12)
    plt.title('GPU加速比', fontsize=14)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f'\n观察: 矩阵越大,GPU加速效果越明显!')
else:
    print('需要GPU才能进行性能对比')

---

## 第五部分: 神经网络与GPU

### 5.1 在GPU上创建和训练模型

In [ ]:
# 定义一个简单的MLP
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = x.view(-1, 784)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

# 创建模型
net = SimpleMLP()
print(f'模型参数所在设备(创建时): {next(net.parameters()).device}')

# 移动到GPU
device = try_gpu()
net = net.to(device)
print(f'模型参数所在设备(移动后): {next(net.parameters()).device}')

### 5.2 在GPU上进行前向传播

In [ ]:
# 创建输入数据并移动到GPU
X = torch.rand(4, 1, 28, 28).to(device)
print(f'输入数据在: {X.device}')

# 前向传播
output = net(X)
print(f'输出在: {output.device}')
print(f'输出形状: {output.shape}')

### 5.3 完整的训练循环示例

In [ ]:
import torchvision
from torch.utils.data import DataLoader
from torchvision import transforms

# 准备数据
transform = transforms.ToTensor()
train_dataset = torchvision.datasets.FashionMNIST(
    root='../data', train=True, transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

# 模型、损失、优化器
device = try_gpu()
net = SimpleMLP().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.1)

print(f'训练设备: {device}')
print(f'数据批次数: {len(train_loader)}')

In [ ]:
# 训练一个epoch
def train_epoch(net, train_loader, loss_fn, optimizer, device):
    net.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for X, y in train_loader:
        # 数据移动到设备
        X, y = X.to(device), y.to(device)
        
        # 前向传播
        pred = net(X)
        loss = loss_fn(pred, y)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 统计
        total_loss += loss.item() * X.size(0)
        correct += (pred.argmax(1) == y).sum().item()
        total += X.size(0)
    
    return total_loss / total, correct / total

# 训练3个epoch
num_epochs = 3
print('开始训练...')
start_time = time.time()

for epoch in range(num_epochs):
    loss, acc = train_epoch(net, train_loader, loss_fn, optimizer, device)
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss:.4f}, Acc: {acc:.4f}')

elapsed = time.time() - start_time
print(f'\n训练完成! 总用时: {elapsed:.2f}秒')
print(f'平均每epoch: {elapsed/num_epochs:.2f}秒')

### 5.4 CPU vs GPU训练速度对比

In [ ]:
if torch.cuda.is_available():
    # 小数据集用于快速测试
    small_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    
    # CPU训练
    print('CPU训练...')
    net_cpu = SimpleMLP().to('cpu')
    optimizer_cpu = torch.optim.SGD(net_cpu.parameters(), lr=0.1)
    
    start = time.time()
    loss_cpu, acc_cpu = train_epoch(net_cpu, small_loader, loss_fn, optimizer_cpu, 'cpu')
    cpu_time = time.time() - start
    print(f'  时间: {cpu_time:.2f}秒, Loss: {loss_cpu:.4f}, Acc: {acc_cpu:.4f}')
    
    # GPU训练
    print('\nGPU训练...')
    net_gpu = SimpleMLP().to('cuda')
    optimizer_gpu = torch.optim.SGD(net_gpu.parameters(), lr=0.1)
    
    start = time.time()
    loss_gpu, acc_gpu = train_epoch(net_gpu, small_loader, loss_fn, optimizer_gpu, 'cuda')
    torch.cuda.synchronize()
    gpu_time = time.time() - start
    print(f'  时间: {gpu_time:.2f}秒, Loss: {loss_gpu:.4f}, Acc: {acc_gpu:.4f}')
    
    print(f'\n加速比: {cpu_time/gpu_time:.2f}x')
else:
    print('需要GPU才能进行训练速度对比')

---

## 第六部分: 多GPU训练基础

### 6.1 数据并行(DataParallel)

In [ ]:
if torch.cuda.device_count() > 1:
    print(f'使用 {torch.cuda.device_count()} 个GPU训练')
    
    # 数据并行
    net = SimpleMLP()
    net = nn.DataParallel(net)  # 自动分配到所有可用GPU
    net = net.cuda()
    
    print('模型已包装为DataParallel')
    print(f'使用的GPU: {net.device_ids}')
    
elif torch.cuda.device_count() == 1:
    print('只有1个GPU,DataParallel不会带来加速')
else:
    print('没有GPU,无法演示DataParallel')

**DataParallel的工作原理**:
1. 将模型复制到所有GPU
2. 将batch分割到不同GPU
3. 每个GPU独立前向传播
4. 收集所有GPU的输出计算损失
5. 反向传播,梯度在GPU 0上汇总
6. 更新GPU 0的参数,然后同步到其他GPU

---

## 第七部分: 最佳实践

### 7.1 常见陷阱

**陷阱1: 忘记移动数据**

In [ ]:
if torch.cuda.is_available():
    net = SimpleMLP().to('cuda')
    X = torch.rand(4, 1, 28, 28)  # 在CPU上
    
    try:
        output = net(X)  # 错误!数据在CPU,模型在GPU
    except RuntimeError as e:
        print(f'错误: {e}')
    
    # 正确做法
    X = X.to('cuda')
    output = net(X)
    print('\n正确: 数据和模型都在GPU上')

**陷阱2: 频繁的CPU-GPU数据传输**

In [ ]:
if torch.cuda.is_available():
    # 不好的做法: 循环中频繁传输
    X = torch.rand(1000, 100)
    W = torch.rand(100, 10).cuda()
    
    start = time.time()
    for i in range(100):
        x = X[i].cuda()  # 每次都传输
        y = torch.matmul(x, W)
    torch.cuda.synchronize()
    bad_time = time.time() - start
    
    # 好的做法: 一次性传输
    X = X.cuda()
    
    start = time.time()
    for i in range(100):
        y = torch.matmul(X[i], W)
    torch.cuda.synchronize()
    good_time = time.time() - start
    
    print(f'频繁传输: {bad_time:.4f}秒')
    print(f'一次传输: {good_time:.4f}秒')
    print(f'提速: {bad_time/good_time:.2f}x')

### 7.2 标准训练模板

In [ ]:
# 标准的GPU训练代码模板
def train_model_template():
    # 1. 设置设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')
    
    # 2. 创建模型并移动到设备
    model = SimpleMLP().to(device)
    
    # 3. 定义损失和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters())
    
    # 4. 训练循环
    for epoch in range(num_epochs):
        for batch_idx, (data, target) in enumerate(train_loader):
            # 移动数据到设备
            data, target = data.to(device), target.to(device)
            
            # 前向传播
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            
            # 反向传播
            loss.backward()
            optimizer.step()
    
    return model

print('训练模板代码已定义')

---

## 小结

### 核心要点

1. **GPU vs CPU**:
   - GPU适合大规模并行计算
   - 深度学习训练可加速10-100倍
   - 矩阵越大,加速效果越明显

2. **设备管理**:
   ```python
   device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
   ```

3. **张量操作**:
   - 创建: `torch.rand(3, 4, device='cuda')`
   - 移动: `tensor.to(device)` 或 `tensor.cuda()`
   - 查询: `tensor.device`

4. **模型训练**:
   - 模型移动: `model.to(device)`
   - 数据移动: `data.to(device)`
   - 记得同步: `torch.cuda.synchronize()`

5. **多GPU**:
   - 数据并行: `nn.DataParallel(model)`
   - 自动分配到所有GPU

### 最佳实践

✅ **DO**:
- 尽早将模型和数据移动到GPU
- 使用`.to(device)`实现设备无关代码
- 批量传输数据而不是逐个传输
- 监控GPU内存使用(nvidia-smi)

❌ **DON'T**:
- 频繁的CPU-GPU数据传输
- 在循环中反复移动数据
- 不同设备的张量直接运算
- 忘记调用`.to(device)`

### 调试技巧

```python
# 检查设备
print(f'Model device: {next(model.parameters()).device}')
print(f'Data device: {data.device}')

# 查看GPU内存
print(f'Memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print(f'Memory cached: {torch.cuda.memory_reserved()/1e9:.2f} GB')

# 清理GPU内存
torch.cuda.empty_cache()
```

### 性能提示

1. **使用更大的batch_size**: GPU并行能力强
2. **使用混合精度训练**: `torch.cuda.amp`可提速2-3倍
3. **使用DataLoader的pin_memory**: 加速数据传输
   ```python
   DataLoader(..., pin_memory=True)
   ```
4. **使用多workers加载数据**:
   ```python
   DataLoader(..., num_workers=4)
   ```

## 练习

1. **性能测试**: 测试不同batch_size对GPU利用率的影响
2. **内存监控**: 编写代码监控训练过程中的GPU内存使用
3. **多GPU训练**: 使用DataParallel在多GPU上训练
4. **混合精度**: 实现FP16混合精度训练
5. **CPU-GPU对比**: 在完整的CIFAR-10数据集上对比训练速度
6. **GPU利用率**: 使用nvidia-smi监控GPU利用率
7. **内存优化**: 实现梯度累积来训练更大的batch
8. **设备无关代码**: 编写可在CPU和GPU上无缝切换的训练代码